# 07 · API da ANA (HidroWebService): níveis e chuva a cada 15 min

**Antes de rodar:** crie o arquivo `.env` na **raiz do projeto** (ele já está no `.gitignore`; nunca o envie ao GitHub):

```
ANA_ID=seu_cpf_somente_numeros
ANA_SENHA=sua_senha
```

O código em `src/ana_api.py` reaproveita o token (vale 60 min) e só autentica de novo quando precisa; o manual avisa que
autenticações em excesso podem bloquear o IP.

In [ ]:
import sys
sys.path.append("..")
import pandas as pd, matplotlib.pyplot as plt
from src import config, data, ana_api
plt.rcParams["figure.figsize"] = (11, 3.4); plt.rcParams["axes.grid"] = True
cli = ana_api.AnaClient()
print("cliente criado (as credenciais foram lidas do .env)")

## Passo 1 · Teste de conexão (exemplo oficial do manual; não precisa de parâmetro de data)

In [ ]:
df = cli.series("86510000")           # Muçum: últimos 30 dias (janela padrão)
print(len(df), "leituras |", df["Data_Hora_Medicao"].min(), "->", df["Data_Hora_Medicao"].max())
display(df.head(3)); display(df.tail(3))
print("intervalo típico:", df["Data_Hora_Medicao"].diff().median())

**Se funcionou:** a API respondeu e o formato é o esperado (`Cota_Adotada` em cm, `Chuva_Adotada` em mm, `*_Status`: 0 ok, 1 suspeito, 2 ruim).
Compare o horário e o nível da última leitura com o site do SGB (sace.sgb.gov.br) para conferir **fuso** e **unidade**.

**Se deu erro 401/403:** confira ID/senha no `.env`. **Se deu erro de rede ou 5xx:** tente de novo mais tarde.

## Passo 2 · Nome do parâmetro de data (única coisa que o manual não mostra)

No Swagger (https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html): `HidroinfoanaSerieTelemetricaAdotada` → *Try it out*.
Veja a lista de campos e anote o **nome exato** do campo de data (o manual só diz que a "Data de Busca" é obrigatória na rota "detalhada").
Cole abaixo (e mande para mim a lista de campos, se tiver dúvida):

In [ ]:
print("formato aceito:", cli.names_key)
print(cli.names)

## Passo 3 · Teste de uma janela histórica: a cheia de setembro de 2023 em Muçum

In [ ]:
caminho = cli.download_history("86510000", "2023-08-25", "2023-09-15", out_dir=config.ANA_DIR / "_teste")
t = pd.read_csv(caminho, parse_dates=["Data_Hora_Medicao"]).set_index("Data_Hora_Medicao")
print(len(t), "leituras |", t.index.min(), "->", t.index.max())
print("cota máxima:", t["Cota_Adotada"].max(), "cm em", t["Cota_Adotada"].idxmax())
fig, ax = plt.subplots(); t["Cota_Adotada"].plot(ax=ax, lw=0.8); ax.set_ylabel("cota (cm)"); ax.set_title("Muçum, set/2023 (API)"); plt.show()

**Conferência:** nos dados diários do HidroWeb, a maior **média diária** de Muçum foi 2611 cm em 5/9/2023. Aqui o máximo **a cada 15 min**
deve ficar na mesma casa (igual ou maior) e no mesmo dia. Se o dia/horário vier deslocado em algumas horas, é o fuso: ajuste `UTC_OFFSET_HOURS` em `config.py`.

## Passo 4 · Histórico completo (no **terminal**)

São centenas de requisições; rode no terminal e deixe aberto. É **retomável** (grava `data/raw/ana/<codigo>_telemetrica_api.csv`):

```bash
cd ~/projetos/capstone
conda activate capstone
python -m src.ana_api --date-param NOME_NO_SWAGGER --start 2015-01-01 --end 2026-06-30
```

Se alguma estação começou a transmitir depois, as janelas anteriores vêm vazias (é normal). Quando terminar, o notebook 01 já lê esses arquivos.

In [ ]:
# depois do download: conferir cobertura pelo mesmo leitor do pipeline
panel = data.build_panel(raw_dir=config.ANA_DIR)
display(data.coverage_report(panel))